In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("C:/Users/gokup/Downloads/IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
df.shape

(50000, 2)

In [4]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [5]:
df.duplicated().sum()

np.int64(418)

In [6]:
df.drop_duplicates(inplace=True)

In [7]:
df.shape

(49582, 2)

## Text Pre Processing 

### 1. Converting to lower case

In [8]:
df["review"]= df["review"].str.lower()

In [9]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


### 2. Removing the URLs

In [10]:
import re 

def remove_urls(text): 
    text = re.sub(r"http\S+", "",text)
    return text

df["review"]= df["review"].apply(remove_urls)

### 3. Removing Punctuation 

In [11]:
def remove_punctuation(text): 
    text = re.sub(r"[^A-Za-z0-9\s]", "", text)
    return text

df["review"] = df["review"].apply(remove_punctuation)

### 4. Removing html tags 

In [12]:
def remove_tags(text):
    text = re.sub(r"<.*?>","",text) 
    return text

df["review"] = df["review"].apply(remove_tags)

In [13]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


### 5. Removing Stopwords 

In [14]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\gokup\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\gokup\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\gokup\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [15]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords


def remove_stopword(text):
    tokens = word_tokenize(text)
    stopword = stopwords.words("english")

    for word in tokens:
        if word in stopword:
            text = text.replace(word,"")

    return text 

df["review"] = df["review"].apply(remove_stopword)

In [16]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


### 6. Stemming 

In [17]:
from nltk.stem import PorterStemmer

def stemming(text):
    ps = PorterStemmer() 
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens: 
        stemmed_tokens = ps.stem(token)
        stemmed_words.append(stemmed_tokens)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [18]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


### 7. Encoding 

In [19]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])
y = df["sentiment"]

In [20]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

### 8. Vectorization

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)
X = tf.fit_transform(df["review"])

## Tensor Dataset / Data Loaders 

In [39]:
from sklearn.model_selection import train_test_split

x_train , x_test , y_train , y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42)

In [40]:
x_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3248304 stored elements and shape (39665, 5000)>

In [41]:
x_train = x_train.toarray()
x_test = x_test.toarray()

In [32]:
import torch 
from torch.utils.data import DataLoader , TensorDataset , Dataset

train_set = TensorDataset(
    torch.from_numpy(x_train).float(), 
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(x_test).float(), 
    torch.from_numpy(y_test.values).float()
)

In [35]:
train_loader = DataLoader(train_set,batch_size=64,shuffle=True)
test_loader = DataLoader(test_set,batch_size=64)

## Build Our RNN

In [50]:
import torch.nn as nn 
import torch.optim as optim

class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0) 
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [53]:
input_size = x_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

## Train Our Model 

In [54]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction
        
        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.19710655510425568
epoch = 2/10 and loss = 0.25096067786216736
epoch = 3/10 and loss = 0.38205552101135254
epoch = 4/10 and loss = 0.18215422332286835
epoch = 5/10 and loss = 0.32095566391944885
epoch = 6/10 and loss = 0.2789692282676697
epoch = 7/10 and loss = 0.32336705923080444
epoch = 8/10 and loss = 0.4643646478652954
epoch = 9/10 and loss = 0.18098841607570648
epoch = 10/10 and loss = 0.2559938132762909


In [55]:
model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 85.7719068266613
